# Assignment - Applications of Big Data (COMP3002)

__Due: EOD Monday, 16th june 2025 AEDT 23:59__


__Centre for Research in Mathematics And Data Science__

__School of Computer, Data and Mathematical Sciences__


## Description
For this assignment, you will need to create a complete program to perform sentiment classification for movie reviews from feature extraction to classification. For a given review, your program should be able to predict whether it is positive i.e. like the movie, or negative, i.e. dislike the movie. 

## About the data
You will use a large movie review dataset containing a set of 25,000 movie reviews for training, and 25,000 for testing. You can download the data from the vUWS under the assignments folder named aclImdb.zip. You can also visit the following website for more information about the dataset at http://ai.stanford.edu/~amaas/data/sentiment/ or download data directly from there. 

Unzip the data to your local directory. Enter the aclImdb/ directory created by the zip file (~500MB), you will find the following three items among others

- train/: feature files and raw text files for the training set
- test/: feature files and raw text files for the testing set
- README: the readme file for more information on the dataset

Read the README file carefully about the descriptions on the text files that contain the reviews and their naming convention. The directories we concern here are 

- ./aclImdb/train/pos: raw text files of positive reviews in the training set
- ./aclImdb/train/neg: raw text files of negative reviews in the training set
- ./aclImdb/test/pos: raw text files of positive reviews in the test set
- ./aclImdb/test/neg: raw text files of negative reviews in the test set

A full version of this data set is available at hdfs://hadoop.cdms.westernsydney.edu.au:9000/users/bigdata/Hadoop/imdb/fullversion. A tiny cut down version with much less number of files (20 each for training and 10 each for test) is also available at hdfs://hadoop.cdms.westernsydney.edu.au:9000/users/bigdata/Hadoop/imdb/tinyversion. The tiny version is for experimenting purpose. 

## Task 1. Feature extraction (15 points)
Use the map reduce model to convert all text data into matrices. Convert _ratings_ to vectors. These will be used for classification in Task 2. Use TF-IDF to vectorise the text files. See previous practical classes and lectures materials for TF-IDF. One step further though is to represent each text file (review) as a very long and sparse vector as the following. Assume `wordslist` is the final list of distinct words contained in all reviews and its length is $D$. Then each review will be a vector of length $D$, with each position associated with a word in `wordlist` and the value being either 0, if the corresponding word is absent in the review, or the word’s TF-IDF. For example, if `wordlist = [‘word1’, ‘word2’, ‘word3’, ‘word4’]` and review 1 contains `word1` and `word4`, then the vector representation of review 1 is [0.1, 0, 0, 0.4] assuming TF-IDF of `word1` and `word4` in review 1 is 0.1 and 0.4 respectively. Note that TF is calculated from one single document while IDF is obtained from all documents in the collection. 

### Requirements: 

1. Map reduce model is a must. Implement it using Hadoop streaming. All data are available on SCDMS HDFS. The recommendation is to work on the tiny version of the data to make the code work. You may try your code on the full version. However, the application to full version is not required. 
2. Generate two matrices: `training_data`, `test_data`, and two vectors, `training_targets`, `test_targets`. `training_data` should have $N$ rows and $D$ columns with each row corresponding to each review in the training set, where $N$ is the totally number of reviews in training set and $D$ is the total number of words. $N$ and $D$ vary depending on which version of the data you use. `training_targets` should have $N$ elements each of which is the rating of the review is for. `test_data` and `test_targets` are similar defined. 

Note:

<!--1.	If feature extraction is too difficult for you, you can use pre-computed bag of words features included in this data set. Refer to the Appendix and README file for details. However, if pre-computed features are used, __a 60% penalty__ will incur for this task, i.e. the maximum marks you can get from this task is 6 if you do so. -->
- Ratings scores extraction can be purely python. 
- Using map reduce model to extract TF-IDF is mandatory. If not used, a __50% penalty__ for this task will incur. There is no constraint on how to form the training and test matrices and vectors. There are many versions of TF-IDF. There is no preference for which version to use.
- You can use data frame (using `pandas` package) instead of matrices and vectors to store training and test data and targets. 

### Marking scheme for task 1:

<!--- Text file reading (1pt): read the text files for TF-IDF extraction. -->
- Rating scores extraction (3pts): parse the name of text files to extract ratings.
- TF-IDF extraction (10pts):  use map reduce model to extract TF-IDF for each text file.  
- Forming matrices and target vectors (or data frames) (2pts): collect TF-IDFs to form training and test data for task 2. 




## Task 1: submission 
Your work goes from here. Add blocks when neceesay. Add inline comments in python code or in markdown blocks. 

Here goes my first mapreducer, mapredcue1. 

### 1.1 Rating Score Extraction

The rating score of each review is encoded in its file name using the
format `reviewID_rating.txt`. Python is used to parse the file names and
extract the ratings for the training and test reviews.

In [2]:
import os

# Path to the IMDb dataset
DATA_PATH = "data/aclImdb"

# Define the four review folders
train_pos_path = os.path.join(DATA_PATH, "train", "pos")
train_neg_path = os.path.join(DATA_PATH, "train", "neg")
test_pos_path = os.path.join(DATA_PATH, "test", "pos")
test_neg_path = os.path.join(DATA_PATH, "test", "neg")


def extract_rating(filename):
    """
    Extract the rating score from an IMDb review filename.

    Example:
    '123_9.txt' -> 9
    """
    rating = filename.split("_")[1].split(".")[0]
    return int(rating)


# Check the function using one real review file
sample_filename = sorted([
    filename
    for filename in os.listdir(train_pos_path)
    if filename.endswith(".txt")
])[0]

print("Sample file from the dataset:", sample_filename)
print("Extracted rating:", extract_rating(sample_filename))

Sample file from the dataset: 0_9.txt
Extracted rating: 9


### 1.2 Review Metadata and Target Vectors

This section reads the file names from the four review folders:

- training positive reviews
- training negative reviews
- test positive reviews
- test negative reviews

For each review, the program records its file name, rating, dataset split,
sentiment folder and file path.

The ratings are then collected into `training_targets` and `test_targets`.
These are the target vectors required for Task 1.

In [4]:
import pandas as pd


# Create an empty list.
# Each review will later be stored as one dictionary inside this list.
all_reviews = []


def add_reviews_from_folder(folder_path, split_name, sentiment_name):
    """
    Read review file names from one folder
    and add their information to all_reviews.

    Example folder:
    data/aclImdb/train/pos
    """

    # Check that the folder exists
    if not os.path.isdir(folder_path):
        raise FileNotFoundError(
            "Folder not found: " + folder_path
        )

    # Get all file names from the folder
    file_names = os.listdir(folder_path)

    # Sort the names so that the order stays consistent
    file_names = sorted(file_names)

    # Read one file name at a time
    for file_name in file_names:

        # Only use text files
        if not file_name.endswith(".txt"):
            continue

        # Extract the rating from the file name
        rating = extract_rating(file_name)

        # Create a unique ID for the review
        document_id = (
            split_name
            + "/"
            + sentiment_name
            + "/"
            + file_name
        )

        # Create the complete local file path
        full_file_path = os.path.join(
            folder_path,
            file_name
        )

        # Store the information of one review
        review_information = {
            "document_id": document_id,
            "split": split_name,
            "sentiment_folder": sentiment_name,
            "filename": file_name,
            "rating": rating,
            "file_path": full_file_path
        }

        # Add this review to the main list
        all_reviews.append(review_information)


# Read the four labelled review folders
add_reviews_from_folder(
    train_pos_path,
    "train",
    "pos"
)

add_reviews_from_folder(
    train_neg_path,
    "train",
    "neg"
)

add_reviews_from_folder(
    test_pos_path,
    "test",
    "pos"
)

add_reviews_from_folder(
    test_neg_path,
    "test",
    "neg"
)


# Convert the list into a pandas DataFrame
review_metadata = pd.DataFrame(all_reviews)


# Select training reviews
training_metadata = review_metadata[
    review_metadata["split"] == "train"
].copy()

training_metadata = training_metadata.reset_index(
    drop=True
)


# Select test reviews
test_metadata = review_metadata[
    review_metadata["split"] == "test"
].copy()

test_metadata = test_metadata.reset_index(
    drop=True
)


# Extract the original ratings as target vectors
training_targets = training_metadata[
    "rating"
].to_numpy()

test_targets = test_metadata[
    "rating"
].to_numpy()


# Display simple checks
print("Training reviews:", len(training_metadata))
print("Test reviews:", len(test_metadata))

print()
print("training_targets shape:", training_targets.shape)
print("test_targets shape:", test_targets.shape)

print()
print("First 10 training ratings:")
print(training_targets[:10])

display(training_metadata.head())

Training reviews: 25000
Test reviews: 25000

training_targets shape: (25000,)
test_targets shape: (25000,)

First 10 training ratings:
[ 9  8 10  7  8  8  7  7  7  7]


,document_id,split,sentiment_folder,filename,rating,file_path
0,train/pos/0_9.txt,train,pos,0_9.txt,9,data/aclImdb\train\pos\0_9.txt
1,train/pos/10000_8.txt,train,pos,10000_8.txt,8,data/aclImdb\train\pos\10000_8.txt
2,train/pos/10001_10.txt,train,pos,10001_10.txt,10,data/aclImdb\train\pos\10001_10.txt
3,train/pos/10002_7.txt,train,pos,10002_7.txt,7,data/aclImdb\train\pos\10002_7.txt
4,train/pos/10003_8.txt,train,pos,10003_8.txt,8,data/aclImdb\train\pos\10003_8.txt


### 1.3 MapReduce Job 1: Word Frequency in Each Review

The first MapReduce job counts how many times each word appears in each
review.

The mapper reads the review text and emits one record for every word
occurrence. The reducer groups identical document-word pairs and adds
their values together.

In this implementation, the raw word count is used as the term frequency.

The output of this job contains three fields:

- `document_id`
- `word`
- `term_count`

#### 1.3.1 Mapper: Reading and Tokenising Reviews

The mapper receives the content of each review through standard input.

For every review, the mapper:

1. identifies the source document;
2. converts the text to lowercase;
3. removes HTML line-break markers;
4. extracts individual English words; and
5. emits the value `1` for every word occurrence.

For example, the review `Good movie, good acting!` produces:

    train/pos/0_9.txt|good      1
    train/pos/0_9.txt|movie     1
    train/pos/0_9.txt|good      1
    train/pos/0_9.txt|acting    1

The document ID and word are combined into one key. This allows Hadoop
to group identical document-word pairs before sending them to the
reducer.

In [5]:
%%writefile hadoopmapper1.py
#!/usr/bin/env python3

import os
import re
import sys


def get_document_id():
    """
    Get the ID of the review currently being processed.

    Example input path:
    /users/bigdata/Hadoop/imdb/tinyversion/train/pos/0_9.txt

    Returned document ID:
    train/pos/0_9.txt
    """

    # Get the current input file path from Hadoop
    input_file_path = os.environ.get(
        "mapreduce_map_input_file"
    )

    # Some Hadoop versions use a different variable name
    if input_file_path is None:
        input_file_path = os.environ.get(
            "map_input_file"
        )

    # Use a safe name if Hadoop does not provide the path
    if input_file_path is None:
        return "unknown_file"

    # Replace Windows separators with standard separators
    input_file_path = input_file_path.replace(
        "\\",
        "/"
    )

    # Split the path into individual parts
    path_parts = input_file_path.split("/")

    # Keep the final three parts:
    # train/pos/0_9.txt
    document_id = "/".join(
        path_parts[-3:]
    )

    return document_id


def extract_words(review_text):
    """
    Convert review text into lowercase English words.

    Example:
    'A GREAT movie!' becomes ['a', 'great', 'movie']
    """

    # Convert all letters to lowercase
    review_text = review_text.lower()

    # Remove the common HTML line-break marker
    review_text = review_text.replace(
        "<br />",
        " "
    )

    # Extract sequences containing letters from a to z
    words = re.findall(
        r"[a-z]+",
        review_text
    )

    return words


# Identify the review currently being processed
document_id = get_document_id()


# Hadoop sends review text one line at a time
for line in sys.stdin:

    # Convert the current line into words
    words = extract_words(line)

    # Emit one record for each word occurrence
    for word in words:

        combined_key = document_id + "|" + word

        # Output format:
        # document_id|word    1
        print(
            combined_key + "\t1"
        )

Writing hadoopmapper1.py


#### 1.3.2 Reducer: Counting Words in Each Review

Hadoop sorts the mapper output before sending it to the reducer.
Therefore, identical document-word keys appear next to each other.

The reducer adds the values belonging to the same key.

For example, these mapper records:

    train/pos/0_9.txt|good    1
    train/pos/0_9.txt|good    1

are reduced to:

    train/pos/0_9.txt    good    2

The final value means that the word `good` appears twice in the review.

In [6]:
%%writefile hadoopreducer1.py
#!/usr/bin/env python3

import sys


def print_word_count(combined_key, total_count):
    """
    Print the final count for one document-word pair.

    Example input key:
    train/pos/0_9.txt|good

    Example output:
    train/pos/0_9.txt    good    2
    """

    # Separate the document ID and word
    document_id, word = combined_key.split(
        "|",
        1
    )

    # Output three tab-separated fields
    print(
        document_id
        + "\t"
        + word
        + "\t"
        + str(total_count)
    )


# Store the key currently being counted
current_key = None

# Store the total count for the current key
current_count = 0


# Hadoop sends sorted mapper output one line at a time
for line in sys.stdin:

    # Remove the line-break character and extra spaces
    line = line.strip()

    # Ignore empty lines
    if line == "":
        continue

    # Separate the key and value
    combined_key, count_text = line.split(
        "\t",
        1
    )

    # Convert the count from text into an integer
    count = int(count_text)

    # The current line belongs to the same key
    if combined_key == current_key:

        current_count = current_count + count

    else:

        # Print the completed previous key
        if current_key is not None:

            print_word_count(
                current_key,
                current_count
            )

        # Start counting a new key
        current_key = combined_key
        current_count = count


# Print the final key after the loop ends
if current_key is not None:

    print_word_count(
        current_key,
        current_count
    )

Writing hadoopreducer1.py


#### 1.3.3 Hadoop Streaming Command

The following Hadoop Streaming command executes MapReduce Job 1 on the
tiny IMDb dataset stored on SCDMS HDFS.

Recursive input processing is enabled because the reviews are stored in
nested training, test, positive and negative directories.

The previous output directory is removed before execution because
Hadoop cannot overwrite an existing output directory.

The mapper and reducer scripts created above are sent to the Hadoop
workers using the `-file` options.

In [ ]:
%%bash

# Hadoop Streaming library
STREAMING_JAR="/usr/local/hadoop/share/hadoop/tools/lib/hadoop-streaming-3.3.6.jar"

# Tiny IMDb dataset stored on HDFS
INPUT_PATH="/users/bigdata/Hadoop/imdb/tinyversion"

# Output location inside the current user's HDFS directory
OUTPUT_PATH="/users/${USER}/COMP3002/task1_job1_word_counts"


# Create the parent output directory
hdfs dfs -mkdir -p "/users/${USER}/COMP3002"


# Remove the previous output if it exists
hdfs dfs -rm -r -f "${OUTPUT_PATH}"


# Run MapReduce Job 1
hadoop jar "${STREAMING_JAR}" \
    -D mapreduce.input.fileinputformat.input.dir.recursive=true \
    -mapper "python3 hadoopmapper1.py" \
    -reducer "python3 hadoopreducer1.py" \
    -input "${INPUT_PATH}" \
    -output "${OUTPUT_PATH}" \
    -file hadoopmapper1.py \
    -file hadoopreducer1.py


# Display the first 10 output records
echo
echo "First 10 records from MapReduce Job 1:"

hdfs dfs -cat "${OUTPUT_PATH}/part-*" | head -n 10

<hr style="height:4px;border-width:0;color:gray;background-color:green">

## Task 2. Classification (15 points)
Construct a classification model for review sentiment prediction, meaning that given a customer review (taken from test set) about a movie, your program should be able to predict whether it is positive or negative. There is no limitation on how many classifiers and what specific model you should use. You can simply pick one that works for you for this task, either from those covered in lectures and practical classs or any other classifiers from any python packages. A good starting point is the `scikit-learn` (i.e. `sklearn`) package. 
A few things you need to address in your python program are listed as requirements below. 

### Requirements: 

1.	Data pre-processing. In task 1, you have extracted the ratings vectors for training and test. They are raw ratings. As we are interested in sentiment prediction, i.e. to predict either the review is positive or negative. You need to convert all ratings>5 as positive class and ratings<=5 as negative class. Choose a coding scheme, e.g. 1 for positive, 0 for negative. 
2.	Normalisation. Apply at least one normalisation scheme and compare the performance of the classifier(s) with and without normalisation. 
3.	Training and model selection. Use cross validation to select the best parameters for your classifier. There may be many parameters to tune in some classifiers such as random forest classifier (RFC). You can focus on the most important one(s) such as `max_depth` and `n_estimators` in RFC. Refer to `scikit-learn` package documentation for details. 
Hint: you can start with a small subset of training set to test a few parameters to get a feel of what range the parameters should be that make the model perform well in terms of prediction accuracy. Then turn on large scale cross validation on the whole training set.  
4.	Test on test data. After model selection, apply the best model, i.e. model with the parameters that produces the best cross validation scores, to test data and make prediction for each review and record prediction accuracy (ACC). 

Note: 

1. Always train your classifier(s) ONLY on training data including cross validation. After model selection, apply the best model on test data to evaluate the performance. 
2. Good performance, i.e. higher ACC on test data, is not essential for this task. However, if your classifier has ACC low than 60%, it usually means that there are some mistakes somewhere in your code. So try to score as high ACC as possible. 
3.	You are encouraged to try many classifiers. If the coding is right, this should not be too difficult. Remember model selection when you try different classifiers!

### Marking scheme for task 2:

- Data pre-processing (1pts): convert ratings to positive and negative coding scheme. 
- Normalisation and comparison (3pts): apply normalisation and compare performance difference with and without it.
- Training on training data (3pts): training performed on training data.
- Cross validation (6pts): apply cross validation on training data. 
- Testing on test data (2pts): best model applied to test data and ACC produced.  




<hr>

## Task 2: submission 
Your work goes from here. Add blocks when neceesay. Add inline comments in python code or in markdown blocks. 

Here goes my first python script. 

In [ ]:
# Your python code for Task 2: data preprocessing
# Your code goes from here. 



<hr style="height:4px;border-width:0;color:gray;background-color:blue">

## Bonus Task (10 points)
This is a bonus task. It is not essential but if you could complete it as required you will receive 10 extra points towards your final results of this unit. The task is similar to the _Task 5_ in prac 8. 

Compute the correlation between features and response. Use the TF-IDF as features and review scores as response. Consider only training set, i.e. on `training_data`. Here is the details. Let $\mathbf x_i$ be the $i$-th TF-IDF _column_ vector you extracted for the $i$-th review, and $y_i$ its corresponding review score. To compute the coorelation, we need $\tilde{\mathbf x}_i$ and $\tilde{y}_i$, normalised version of $\mathbf x_i$ and $y_i$ as the following. 
$$
\tilde{\mathbf x}_i = \frac{\hat{\mathbf x}_i}{\|\hat{\mathbf x}_i\|} 
$$
where $\hat{\mathbf x}_i = \mathbf x_i - \mathbf m$, $\mathbf m$ is the mean of all features, i.e. $ \mathbf m = \frac{\sum_{i=1}^N \mathbf x_i}N$, and $\|\hat{\mathbf x}_i\|$ is the so-called $\ell_2$ norm of $\|\hat{\mathbf x}_i\|$ which is defined as 
$$\|\hat{\mathbf x}_i\| = \sqrt{\sum_{j=1}^Dx_{i_j}^2}$$
i.e. the square root of the sum of squares of all the elements in vector $\hat{\mathbf x}_i$. $\tilde{y}_i$ is similar 
$$ \tilde{y}_i = \frac{y_i}{\|\mathbf y\|}$$
where $\mathbf y=[y_1,\ldots,y_N]$ is the vector of all review scores. Then the correlation $\mathbf r$ is  
$$
\mathbf r = \sum_{i=1}^N\tilde{y}_i\tilde{\mathbf x}_i.
$$
$\mathbf r$ will be a vector of length $D$. 

### Requirements: 

1. Use map reduce computing model for this task is mandatory. Direct computing the correlation from the matrices obtained from Task 1, i.e. `training_data` is _not_ acceptable. 

2. Python code and Hadoop streaming commands must be supplied for this taks. If multiple map reduce steps are used, a step-by-step guidance must be provided as well. 


_Hint: you may consider several map reduce to compute mean, $\ell_2$ norm, multiplication and etc._

<hr>

## Bonus task : submission 
Your work goes from here. Add blocks when neceesay. Add inline comments in python code or in markdown blocks. 

Here goes my first mapreducer, task3mapredcue1. 

In [ ]:
# Your python code for mapreduce1
# Assuming you use separate python source files for mapper and reducer. 
# save as task3hadoopmapper1.py


# save as task3hadoopreducer1.py



Run the following script in hadoop with the above python source files saved. 

In [ ]:
%%bash
hadoop jar /usr/local/hadoop/share/hadoop/tools/lib/hadoop-streaming-3.3.6.jar \
  -mapper task3hadoopmapper1.py \
  -reducer task3hadoopreducer1.py \
  -input /users/bigdata/imdb/tinyversion \
  -output /hdfs/path/to/outputdirectory \
  -file local/path/to/task3hadoopmapper1.py \
  -file local/path/to/task3hadoopreducer1.py

Here goes my second mapreducer, task3mapredcue2. 

In [ ]:
# Your python code for task3mapreduce1
# Assuming you use a single python source file for mapper and reducer. 
# save as task3hadoopmapreducer2.py





Run the following script in hadoop with the above python source file saved. 

In [ ]:
%%bash
hadoop jar /usr/local/hadoop/share/hadoop/tools/lib/hadoop-streaming-3.3.6.jar \
  -mapper 'python task3hadoopmapreducer2.py map' \
  -reducer 'python task3hadoopmapreducer2.py reduce' \
  -input /users/bigdata/imdb/tinyversion \
  -output /hdfs/path/to/outputdirectory \
  -file local/path/to/task3hadoopmapreducer2.py 

<hr style="height:4px;border-width:0;color:red;background-color:red">

## Overall Marking Criteria for All Tasks

Your program will be marked against both functional and operational requirements. Functional requirements accounts for 80% of the mark, which measure how well your program achieves the expected functionalities and are further broken down into the items listed in marking schemes in those tasks. 

In addition to function requirements, your program should also meet the operational and style requirements, which can be broken down into the following. 

- Readability (5%): Comments should be included in your program to explain the main idea of your design; use meaningful variable and function names; do not declare variables that are not used in the program 
- Modularity (10%): Your program should make use of functions or classes wherever possible to achieve modular design and maximise reusability. 
- Useability (5%): Your program should be easy to use by the user. These include displaying messages for user interaction, performing adequate input validation, and allowing the users to choose the locations of the data file for model training. 

## Submission
Your python code solution including Hadoop streaming commands if any, documentation such as how to use the functions must be written in _this jupyter notebook_ with clear indication which is for which. Rename it to `COMP3002_assignment_yourstudentid.ipynb` and work on it. Add code and markdown blocks as you like. Submit your complete notebook through vUWS before deadline.